# Sleep Staging on the Sleep Physionet dataset


## Libraries and Config

In [ ]:
 # this ensures that plots open in a new window
%matplotlib qt

In [ ]:
import os
import copy

import mne
import torch
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pathlib
from tqdm.notebook import tqdm
from tqdm.contrib.concurrent import thread_map

matplotlib.use("QtAgg") # use the Qt backend for interactive plotting
mne.set_log_level("ERROR")  # suppress MNE info and warnings for cleaner output

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print("Using GPU for computations. Training should be faster.")
else:
    print(
        "No GPU found. Using CPU for computations, training might be slower."
        "\n\nIf running on Google Colab, make sure to enable GPU acceleration in the notebook settings."
    )

## Loading the data

In [ ]:
from mne.datasets.sleep_physionet.age import fetch_data

subjects=range(83) # fetch data for all 83 subjects
recording=[1, 2] # fetch both recordings for each subject,
sleep_physionet_data_path= pathlib.Path("./data")  # specify the path where you want to store the data

sleep_physionet_data_fnames = fetch_data(
    subjects=subjects,
    recording=recording,
    path=sleep_physionet_data_path,
    on_missing="warn"
)

len(sleep_physionet_data_fnames)

In [ ]:
from src.utils import load_sleep_physionet_raw_data

# load all raw data
raws = [
    load_sleep_physionet_raw_data(
        raw_fname=fname[0],
        annot_fname=fname[1]
    ) for fname in sleep_physionet_data_fnames
]

In [ ]:
# sanity check: plot the first raw data to verify it loaded correctly
raws[0].plot()

## Preprocessing

During the **Awake** stage, our brain primarily produces **Beta waves** (13-30 Hz). As we drift into deeper sleep, these frequencies slow down, eventually reaching **Delta waves** (0.5-4 Hz) in N3/N4 deep sleep. Even during REM sleep, the brain typically produces **Theta waves** (4-8 Hz). This means that for sleep staging, the most relevant EEG information lies below 30 Hz.

Therefore, we will apply a simple **low-pass filter** with a cutoff frequency of 30 Hz. This removes higher-frequency noise (like muscle artifacts or line noise) that could negatively impact our model's performance.

In [ ]:
l_freq, h_freq = None, 30

filter_dir = sleep_physionet_data_path / "sleep_physionet_data_filtered"
filter_dir.mkdir(exist_ok=True)

for raw in tqdm(raws, desc="Filtering raw data and saving filtered files"):
    # Load data into memory before filtering (required by MNE)
    raw.load_data().filter(l_freq=l_freq, h_freq=h_freq)
    # save the filtered raw data to a new file
    raw.save(
        fname= filter_dir / f"{raw.filenames[0].stem}_filtered_raw.fif",
        overwrite=True
    )

del raws  # free up memory by deleting the original raw objects

In [ ]:
# let's read the filtered raw data back to verify it was saved correctly and to free up memory from the original raw objects
raws_filtered = []

filtered_raw_files = list(filter_dir.glob("*_filtered_raw.fif"))
for raw_file in filtered_raw_files:
    raw = mne.io.read_raw_fif(raw_file, preload=False) # preload=False to avoid loading all data into memory at once
    raws_filtered.append(raw)

As a sanity check, let's plot the Power Spectral Density (PSD) of one of the raw recordings.

> ### What is PSD?
> In simple terms, PSD is a static overview that shows the "strength" or "power" of each frequency in the entire signal. Since we applied a 30 Hz low-pass filter, we expect to see a significant drop in power for frequencies above 30 Hz in the plot below.

In [ ]:
raws_filtered[0].plot_psd()

### Creating Epochs Dataset

In [ ]:
from src.utils import extract_epochs, scale_epoch
from src.datasets import EpochsDataset

from torch.utils.data import ConcatDataset

from tqdm.notebook import tqdm

#  EEG epoch window length
eeg_epoch_duration = 30.0

all_datasets = []  # list to hold the datasets for all subjects and recordings
for raw in tqdm(raws_filtered, desc="Processing raw data and creating datasets"):
    # get the epochs objects
    epochs = extract_epochs(raw=raw, epoch_length=eeg_epoch_duration)

    # extract the data arrats needed for the Dataset class
    # Note: We copy them to ensure they are standard NumPy arrays (not memory maps).
    X = epochs.get_data(copy=True)  # shape: (n_epochs, n_channels, n_times)
    y = epochs.events[:, -1] # shape: (n_epochs,), The last/third column contains the event IDs
    
    # get metadata
    subject_id = raw.info["subject_info"]["id"]
    recording_id = int(raw.info["subject_info"]["his_id"].split(" ")[-1])

    # create the dataset
    epoch_ds = EpochsDataset(
        epochs_data=X,
        epochs_labels=y,
        subject_id=subject_id,
        recording_id=recording_id,
        transform=scale_epoch
    )
    all_datasets.append(epoch_ds) # add the dataset to the list of all datasets

# concatenate all datasets into one
dataset = ConcatDataset(all_datasets)
print(f"Total number of epochs in the combined dataset: {len(dataset)}")

## Making train, valid and test splits
To strictly avoid data leakage, we perform a subject-wise split or subject-aware splitting. Sleep patterns are highly individualistic; if a model sees Subject A's Recording 1 in the training set, it will perform artificially well on Subject A's Recording 2 in the test set. By ensuring all recordings from a specific subject are confined to strictly one split (Train, Val, or Test), we measure the model's ability to generalize to new, unseen people, which is the gold standard for clinical applications.

In [ ]:
from src.datasets import split_by_subject

# calculate number of subjects for splits (e.g., 20% Test, 20% Val)
total_subjects = len(
    np.unique(
        [ds.subject_id for ds in dataset.datasets]
    )
)
n_subjects_test = max(1, int(total_subjects * 0.2))
n_subjects_val = max(1, int(total_subjects * 0.2))

# perform split
train_ds, valid_ds, test_ds = split_by_subject(
    dataset=dataset,
    n_subjects_test=n_subjects_test,
    n_subjects_val=n_subjects_val
)

del dataset

# Note: the epochs here means different chucks of that particular dataset / iteration
print(f"Train size: {len(train_ds)} epochs")
print(f"Val size: {len(valid_ds)} epochs")
print(f"Test size: {len(test_ds)} epochs")

### Class Imbalance:
Sleep stages are not equal. We spend much more time in N2 than N1. To prevent the model from just guessing "N2" all the time, we compute Class Weights. These weights will be passed to our Loss Function to make the model "pay more attention" to rare classes like N1 and REM.

In [ ]:
import pandas as pd


classes_mapping  = {
    1: "Sleep stage W", 
    2: "Sleep stage 1", 
    3: "Sleep stage 2", 
    4: "Sleep stage 3", 
    5: "Sleep stage R"}

# get all the labels from the training set
train_y = np.concatenate(
    [ds.epochs_labels for ds in train_ds.datasets]
)

# plot the distribution
ax = pd.Series(train_y).map(classes_mapping).value_counts().plot(kind="barh")
ax.set_xlabel("Number of training example")
ax.set_ylabel("Sleep Stage")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# calculate weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_y),
    y=train_y
)

# convert to a PyTorch tensor for the Loss Function later
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

for i, class_weight in enumerate(class_weights):
    print(f"{classes_mapping[i + 1]} : {class_weight}")

## Creating the Neural Network (Architecture)

In this section, we define our Convolutional Neural Network (ConvNet) architecture.

We will use the architecture proposed by **Chambon et al. (2018)**. This specific network is highly regarded because it is specifically designed for multivariate EEG time-series data. It is lightweight, fast to train, and interpretable.

Before looking at the PyTorch code, let's break downn the intuition behind this architecture and the specific techniques it uses.

<div style="text-align: center;"> 
<img src="imgs/chambon_convnet.png" alt="Chambon 2018 Architecture Diagram" width="800"> <br /> Source: <a href="https://github.com/hubertjb/dl-eeg-tutorial/blob/main/sleep_staging_physionet.ipynb">Adapted from Banville et al. 2020 (Tutorial on Deep Learning on Sleep Data)</a> 
</div>

### The Input and Output
+ **Input:** A 30-second window (epoch) of EEG data. For our Physionet dataset, we use $C=2$ channels (sampled at 100 Hz), resulting in an input shape of `(1, 2, 3000)` - where `1` is a dummy "depth" channel required by PyTorch Conv2D layers, `2` is the number of EEF channels, and `3000` is the number of time samples.
+ **Output:** A 5-dimensional vector containing the probability for each of the 5 sleep stages - W, N1, N2, N3, R.

### The Chambon ConvNet Layers
The Chambon architecture is clever because it splits the feature extraction into two distinct steps: **Spatial** (across different electrodes on the head) and **Temporal** (across time).

#### 1. Spatial Convolution (The "Where")
+ **What it does:** The very first layer is a `Spatial Conv (C, 1)`. Instead of lookin at a chunk of time, this filter looks at one single time point across all $C$ channels simultaneously. For our 100 Hz data, one sample represents a tiny slice of time exactly 10 milliseconds long ($1s / 100$). This filter combines the voltage readings from all electrodes at that exact 10 ms instant before moving to the next sample.

+ **The Intuition:** Imagine you have electrodes at the front (Fpz) and back (Pz) of the head. Sometimes, a sleep event is best detected by looking at the *difference* or *combination* of signals from these two locations at the exact same millisecond. This layer acts like a "virtual electrode", learning the optimal linear combination of the physical chaneels to highlight sleep-relevant brain activity. 


#### 2. Permute (Reshaping)
+ **What is does:** This is a simple matrix transpose operation. It swaps the dimensions of the data tensor so that the newly created "virtual channels" from the spatial convolution are ready to be analysed over time.


#### 3. Temporal Convolutions (The "When")
+ **What it does:** Next are the `Temporal Conv (1, 50)` layers. These filters look at a single virtual channel but scan across a window of time (e.g., 50 time sample, which is 0.5 seconds at 100 Hz).

+ **The Intuition:** This is where the network looks for specific wave shapes or transient events. Fo example, it might learn a filter that perfectly matches the shape of a **Sleep Spindle (11-16 Hz oscillation)** or a **K-complex (sharp slow waves)**. Because of *translation invairance* (i.e., the property of a system, model, or function that produces the same output regardless of a shift in the position of the input data), it can find tese events no matter where they occur in the 30-second window.

#### 4. Non-Linearity: ReLU
+ **What it does:** After the temporal convolutions, the data passes through a ReLU (Rectified Linear Unit) activation function: $f(x) = \max(0, x)$.

+ **The Intuition:** Convolutions are purely linear math (multiplication and addition). If a network only had linear layers, no matter how deep it was, it would just behave like one giant linear regression model. ReLU introduces non-linearity, allowing the network to learn complex, non-linear boundaries between sleep staes, e.g., if the spindle power is above the thresold,  trigger strongly; otherwise, output zero
<div style="text-align: center;"> 
<img src="imgs/relu.png" alt="ReLU activation function" width="300"> <br /> Source: <a href="https://www.researchgate.net/figure/Graphic-representation-of-the-ReLU-activation-function_fig3_348703101">researchgate</a> 
</div>

#### 4. Max Pooling (Downsampling)
+ **What it does:** `Max pool (1, 13)` slides a window (e.g., 13 samples wide) across the time axis and only keeps the maximum value in that window.

+ **The Intuition:** Once the temporal convolution has detected a feature (like a K-complex), we don't necessarily care exactly which millisecond it happened at; we just care that it did happen in that gerenal timeframe. Max pooling achieves three things:
    1. *Reduces dimensionality* - making the network faster and lighter.
    2. *Provides local translation invariance* - a slight shift in the input doesn't change the pooled output.
    3. *Summarise features* - over longer time scales.
    
#### 6. Flatten & Dropout
+ **Flatten:** Takes the 3D feature maps and unrolls them into a single 1D vector so that can be fed into a standard classifier.

+ **Dropout:** Randomly "turns off" a percentage of neurons during training. This forces the network to not rely too heavily  on any single features (e.g., it can't just memorise one specific channel's noise),acting gas a powerful regularisation technique to prevent overfitting.

#### 7. Fully Connected (Dense) Layer
+ **What it does:** The final step. Every neuron from the flattened feature vector connects to 5 output neurons (our 5 sleep stages).

+ **The Intuition:** This later acts as the final judge. It looks at all the complex spatial-temporal features extracted by the previous layers (e.g., "I see hugh Delta power and no eye movement and a K-complex") and weighs them together to make a final decision: "This is 95% likely to be Stage N2 sleep."

In [ ]:
from src.models import SleepStagerChambon2018

# sampling rate
sfreq = raws_filtered[0].info['sfreq']
# number of EEG channels
n_channels = len(raws_filtered[0].ch_names)
# number of 

# Intialise the model
model = SleepStagerChambon2018(
    eeg_epoch_duration=eeg_epoch_duration,
    n_channels=n_channels,
    sfreq=sfreq,
    n_classes=5
)

In [ ]:
# move model to GPU if CUDA is avilable for significantly faster training
print(f"Using device: {device}")
model = model.to(device)